In [1]:
# Loading libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
# Loading the Medicare 2024 dataset and preprocessing it
cols = ["Rndrng_Prvdr_Type", "HCPCS_Cd", "Place_Of_Srvc", "Rndrng_Prvdr_Cntry",
        "Avg_Mdcr_Stdzd_Amt", "Avg_Mdcr_Alowd_Amt"]
df = pd.read_parquet("medicare_2024.parquet", columns=cols)

df = df[(df["Rndrng_Prvdr_Cntry"] == "US") & (df["Avg_Mdcr_Alowd_Amt"] > 0)]
df["log_pay"] = np.log(df["Avg_Mdcr_Stdzd_Amt"].clip(lower=1e-6))
len(df)

9781233

In [2]:
def eta_sq(group_cols):
    group_mean = df.groupby(group_cols, observed=True)["log_pay"].transform("mean")
    total = ((df["log_pay"] - df["log_pay"].mean()) ** 2).sum()
    within = ((df["log_pay"] - group_mean) ** 2).sum()
    return 1 - within / total

groupings = {
    "Type": ["Rndrng_Prvdr_Type"],
    "Type x POS": ["Rndrng_Prvdr_Type", "Place_Of_Srvc"],
    "HCPCS": ["HCPCS_Cd"],
    "HCPCS x POS": ["HCPCS_Cd", "Place_Of_Srvc"],
    "Type x HCPCS x POS": ["Rndrng_Prvdr_Type", "HCPCS_Cd", "Place_Of_Srvc"],
}
eta = pd.Series({name: eta_sq(c) for name, c in groupings.items()}).sort_values()
eta.round(3)

Type                  0.131
Type x POS            0.181
HCPCS                 0.928
HCPCS x POS           0.957
Type x HCPCS x POS    0.975
dtype: float64

In [ ]:
rng = np.random.default_rng(42)

t1 = ["Rndrng_Prvdr_Type", "HCPCS_Cd", "Place_Of_Srvc"]
df["group"] = df.groupby(t1, observed=True).ngroup()
sizes = df["group"].value_counts()

# 200 random big groups (300+ providers) as "ground truth"
big = rng.choice(sizes[sizes >= 300].index, size=200, replace=False)
groups = df[df["group"].isin(big)].groupby("group")["log_pay"]

rows = []
for g, x in groups:
    x = x.to_numpy()
    mad_full = np.median(np.abs(x - np.median(x)))
    if mad_full == 0:
        continue
    for n in [5, 10, 15, 20, 30, 50]:
        s = rng.choice(x, size=(100, n))   # 100 samples of n providers
        mad_n = np.median(np.abs(s - np.median(s, axis=1, keepdims=True)), axis=1)
        rows.append((n, np.median(np.abs(mad_n / mad_full - 1))))

stab = pd.DataFrame(rows, columns=["n", "mad_error"]).groupby("n")["mad_error"].median()
stab.round(3)

n
5     0.492
10    0.323
15    0.271
20    0.228
30    0.182
50    0.142
Name: mad_error, dtype: float64

In [6]:
n_rows = df["group"].map(sizes)
for k in [10, 20, 30, 50]:
    print(f"min {k}: {(n_rows < k).mean():.2%} of rows need a fallback group")

min 10: 1.04% of rows need a fallback group
min 20: 1.77% of rows need a fallback group
min 30: 2.37% of rows need a fallback group
min 50: 3.37% of rows need a fallback group


Minimum peer group size. With 10 providers per group, the MAD(median absolute deviation) is typically off by ~32%, so a row with a true robust z of 3 would typically get a z somewhere between ~2.3 and ~4.4 (3 / 1.32 and 3 / 0.68). With 30 providers the error drops to ~18%.

Raising the threshold from 10 to 30 moves only 1.3% more rows to a fallback group (1.04% to 2.37%), while 97.6% of rows keep the most specific peer group.

Therefore I suggest MIN_PEERS = 30. Since more rows now use fallback groups, the fallback should keep HCPCS (HCPCS x POS), not Type x POS.